# Score calibration — preparing OOF scores for fusion

Before fusion we need both systems to output scores on the **same scale**.

**Current state:**
- Audio E008 (UBM+MAP+aug): threshold ≈ −0.078 → nearly calibrated
- Image E007 (PCA+LogReg+aug): threshold ≈ −5.028 → badly off

If we average raw scores, image dominates just because its numbers are larger —
not because it's more informative. That's wrong.

**Platt calibration** fits a logistic regression on top of OOF scores:
```
calibrated_score = a · raw_score + b
```
where `a` and `b` are learned so that:
- threshold for prior=0.5 becomes exactly 0
- score = 0 means P(target|x) = 0.5
- both systems speak the same log-odds language

**Why OOF scores for calibration?**  
Calibration fitted on training scores would overfit — the model is overconfident
on its own training data. OOF scores are blind predictions, so they represent
the true score distribution the system produces on unseen data.

In [ ]:
from pathlib import Path
import sys, copy
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import librosa
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_curve, auc
from scipy.special import logsumexp
from scipy.stats import norm as scipy_norm
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    "target":    "#E74C3C",
    "nontarget": "#2E86AB",
    "green":     "#27AE60",
    "purple":    "#8E44AD",
    "gray":      "#95A5A6",
    "audio":     "#E67E22",
    "image":     "#8E44AD",
}
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
y_all = manifest["label"].to_numpy()
SEED = 67
print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")

## 1. Collect OOF scores — E007 (image) and E008 (audio)

We re-run both flagship CV loops with their best augmentation configs (+All).
These OOF scores are the honest, unbiased predictions we calibrate on.

In [ ]:
# ---- Image helpers (E007 +All augmentation) ----

def find_png(stem, data_dir):
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".png")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def load_image(path):
    img = np.array(PILImage.open(path).convert("RGB"), dtype=np.float32)
    return img.mean(axis=2).flatten()

def aug_flip(x):       return x.reshape(80,80)[:,::-1].flatten()
def aug_brightness(x, rng): return np.clip(x * rng.uniform(0.7,1.3), 0, 255)
def aug_noise_img(x, rng):  return np.clip(x + rng.normal(0, 15, x.shape), 0, 255)

def load_images_augmented(df, data_dir, seed):
    """Load images with +All augmentation. Returns (X_aug, y_aug)."""
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _, row in df.iterrows():
        orig = load_image(find_png(row["stem"], data_dir))
        for vec in [orig, aug_flip(orig), aug_brightness(orig, rng), aug_noise_img(orig, rng)]:
            X.append(vec); y.append(row["label"])
    return np.stack(X), np.array(y)

def load_images_original(df, data_dir):
    return np.stack([load_image(find_png(row["stem"], data_dir)) for _, row in df.iterrows()])

print("Image helpers ready.")

In [ ]:
# ---- Audio helpers (E008 +All augmentation) ----

def find_wav(stem, data_dir):
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".wav")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def extract_mfcc(y, sr):
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    mfcc   = np.vstack([mfcc, delta, delta2]).T
    mfcc  -= mfcc.mean(axis=0)
    return mfcc

def aug_noise_audio(y, rng, snr_db=20.0):
    p = np.mean(y**2) + 1e-10
    return y + rng.normal(0, np.sqrt(p / 10**(snr_db/10)), len(y)).astype(y.dtype)

def aug_speed(y, rng):
    return librosa.effects.time_stretch(y, rate=rng.uniform(0.9, 1.1))

def extract_audio_augmented(df, data_dir, seed):
    """Extract MFCC frames with +All augmentation. Returns (X_frames, y_frames)."""
    rng = np.random.default_rng(seed)
    all_X, all_y = [], []
    for _, row in df.iterrows():
        wav = find_wav(row["stem"], data_dir)
        y_wav, sr = librosa.load(wav, sr=None, mono=True)
        for y_aug in [y_wav, aug_noise_audio(y_wav, rng), aug_speed(y_wav, rng)]:
            frames = extract_mfcc(y_aug, sr)
            all_X.append(frames)
            all_y.extend([row["label"]] * len(frames))
    return np.vstack(all_X), np.array(all_y)

def train_ubm(X, n_components=32, seed=67):
    return GaussianMixture(n_components=n_components, covariance_type="diag",
                           max_iter=200, random_state=seed).fit(X)

def map_adapt(ubm, X_target, r=16.0):
    log_resp  = ubm._estimate_log_prob(X_target) + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp      = np.exp(log_resp)
    n_k       = resp.sum(axis=0)
    mu_hat    = (resp.T @ X_target) / (n_k[:,None] + 1e-10)
    alpha     = n_k / (n_k + r)
    adapted   = copy.deepcopy(ubm)
    adapted.means_ = alpha[:,None]*mu_hat + (1-alpha[:,None])*ubm.means_
    return adapted

def score_wav(wav_path, adapted, ubm):
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    mfcc  = extract_mfcc(y, sr)
    return float((adapted.score_samples(mfcc) - ubm.score_samples(mfcc)).mean())

print("Audio helpers ready.")

In [ ]:
oof_image = np.full(len(manifest), np.nan)
oof_audio = np.full(len(manifest), np.nan)

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    train_df = manifest.loc[train_idx]
    val_df   = manifest.loc[val_idx]
    fold_seed = SEED + fold_id
    print(f"Fold {fold_id}...")

    # --- Image E007 ---
    X_tr_img, y_tr_img = load_images_augmented(train_df, DATA, fold_seed)
    X_va_img           = load_images_original(val_df, DATA)
    scaler = StandardScaler()
    pca    = PCA(n_components=50, random_state=SEED)
    clf    = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    X_tr_pca = pca.fit_transform(scaler.fit_transform(X_tr_img))
    X_va_pca = pca.transform(scaler.transform(X_va_img))
    clf.fit(X_tr_pca, y_tr_img)
    oof_image[val_idx] = clf.decision_function(X_va_pca)

    # --- Audio E008 ---
    X_tr_aud, y_tr_aud = extract_audio_augmented(train_df, DATA, fold_seed)
    ubm     = train_ubm(X_tr_aud[y_tr_aud==0], seed=SEED)
    adapted = map_adapt(ubm, X_tr_aud[y_tr_aud==1])
    for idx, row in val_df.iterrows():
        oof_audio[idx] = score_wav(find_wav(row["stem"], DATA), adapted, ubm)

    img_eer, _ = compute_eer(oof_image[val_idx][manifest.loc[val_idx,"label"]==1],
                             oof_image[val_idx][manifest.loc[val_idx,"label"]==0])
    aud_eer, _ = compute_eer(oof_audio[val_idx][manifest.loc[val_idx,"label"]==1],
                             oof_audio[val_idx][manifest.loc[val_idx,"label"]==0])
    print(f"  Image EER={img_eer*100:.2f}%  Audio EER={aud_eer*100:.2f}%")

print("\nOOF scores collected.")
print(f"Image score range: [{oof_image.min():.2f}, {oof_image.max():.2f}]")
print(f"Audio score range: [{oof_audio.min():.2f}, {oof_audio.max():.2f}]")

## 2. The calibration problem — visually

Why we can't just average raw scores: the two systems live on completely
different scales. Averaging them gives image a much larger vote.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, scores, label, color, name in [
    (axes[0], oof_audio, y_all, COLORS["audio"], "Audio E008 (raw)"),
    (axes[1], oof_image, y_all, COLORS["image"], "Image E007 (raw)"),
]:
    bins = np.linspace(scores.min(), scores.max(), 40)
    ax.hist(scores[label==0], bins=bins, alpha=0.6, color=COLORS["nontarget"],
            label="non-target", density=True)
    ax.hist(scores[label==1], bins=bins, alpha=0.75, color=COLORS["target"],
            label="target", density=True)
    _, thr = compute_min_dcf(scores[label==1], scores[label==0])
    ax.axvline(0,   color=COLORS["gray"],  ls=":",  lw=1.5, label="ideal 0")
    ax.axvline(thr, color=COLORS["green"], ls="--", lw=2,   label=f"actual thresh={thr:.2f}")
    ax.set_title(name)
    ax.set_xlabel("Raw score")
    ax.legend(fontsize=8)

# Naive average — to show the scale problem
ax = axes[2]
naive = (oof_audio + oof_image) / 2
bins  = np.linspace(naive.min(), naive.max(), 40)
ax.hist(naive[y_all==0], bins=bins, alpha=0.6, color=COLORS["nontarget"],
        label="non-target", density=True)
ax.hist(naive[y_all==1], bins=bins, alpha=0.75, color=COLORS["target"],
        label="target", density=True)
naive_eer, _ = compute_eer(naive[y_all==1], naive[y_all==0])
ax.set_title(f"Naive average (before calib)\nEER={naive_eer*100:.1f}% — image dominates")
ax.set_xlabel("(audio + image) / 2")
ax.legend(fontsize=8)

plt.suptitle("The calibration problem: audio and image live on different scales",
             fontsize=12)
plt.tight_layout()
plt.show()

## 3. Platt calibration

Fit a logistic regression `score_calib = a·score_raw + b` on OOF scores.
With `C=1e6` (no regularization), this is pure Platt calibration — just
learning the two parameters `a` and `b`.

After calibration:
- Both systems output log-odds on the same scale
- Decision threshold = 0 for prior = 0.5
- Scores can be averaged (= equal-weight fusion) or combined with learned weights

In [ ]:
def platt_calibrate(raw_scores: np.ndarray, labels: np.ndarray) -> LogisticRegression:
    """Fit Platt calibration on OOF scores.

    class_weight='balanced' corrects for class imbalance (30 target vs 192 non-target):
    without it, the logistic regression learns the training prior (~13.5% target)
    so decision_function=0 corresponds to P(target)=13.5%, not 50%.
    With balanced weights, decision_function=0 → P(target|x) = 0.5 = Burget's prior.
    """
    cal = LogisticRegression(C=1e6, max_iter=1000, class_weight="balanced")
    cal.fit(raw_scores.reshape(-1, 1), labels)
    return cal

def apply_calibration(cal: LogisticRegression, raw_scores: np.ndarray) -> np.ndarray:
    """Apply calibration. Returns log-odds scores (threshold = 0 at prior 0.5)."""
    return cal.decision_function(raw_scores.reshape(-1, 1))


# Fit calibrators on OOF scores
cal_audio = platt_calibrate(oof_audio, y_all)
cal_image = platt_calibrate(oof_image, y_all)

# Apply calibration
oof_audio_cal = apply_calibration(cal_audio, oof_audio)
oof_image_cal = apply_calibration(cal_image, oof_image)

# Parameters
print("Calibration parameters:")
print(f"  Audio: a={cal_audio.coef_[0,0]:.4f}, b={cal_audio.intercept_[0]:.4f}")
print(f"  Image: a={cal_image.coef_[0,0]:.4f}, b={cal_image.intercept_[0]:.4f}")

print("\nScore ranges after calibration:")
print(f"  Audio: [{oof_audio_cal.min():.2f}, {oof_audio_cal.max():.2f}]")
print(f"  Image: [{oof_image_cal.min():.2f}, {oof_image_cal.max():.2f}]")

# Thresholds after calibration (should be near 0 now)
_, thr_aud_cal = compute_min_dcf(oof_audio_cal[y_all==1], oof_audio_cal[y_all==0])
_, thr_img_cal = compute_min_dcf(oof_image_cal[y_all==1], oof_image_cal[y_all==0])
print(f"\nThresholds after calibration (target: near 0):")
print(f"  Audio: {thr_aud_cal:.4f}")
print(f"  Image: {thr_img_cal:.4f}")

## 4. Before vs after calibration

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for row_idx, (raw, cal, name, color) in enumerate([
    (oof_audio, oof_audio_cal, "Audio E008", COLORS["audio"]),
    (oof_image, oof_image_cal, "Image E007", COLORS["image"]),
]):
    eer_raw, thr_raw = compute_eer(raw[y_all==1], raw[y_all==0])
    eer_cal, thr_cal = compute_eer(cal[y_all==1], cal[y_all==0])

    # Raw scores
    ax = axes[row_idx, 0]
    bins = np.linspace(raw.min(), raw.max(), 35)
    ax.hist(raw[y_all==0], bins=bins, alpha=0.6, color=COLORS["nontarget"],
            label="non-target", density=True)
    ax.hist(raw[y_all==1], bins=bins, alpha=0.75, color=COLORS["target"],
            label="target", density=True)
    ax.axvline(0,       color=COLORS["gray"],  ls=":",  lw=1.5, label="ideal=0")
    ax.axvline(thr_raw, color=COLORS["green"], ls="--", lw=2,   label=f"thresh={thr_raw:.2f}")
    ax.set_title(f"{name} — raw\nEER={eer_raw*100:.1f}%", color=color)
    ax.set_xlabel("Raw score")
    ax.legend(fontsize=7)

    # Calibrated scores
    ax = axes[row_idx, 1]
    bins = np.linspace(cal.min(), cal.max(), 35)
    ax.hist(cal[y_all==0], bins=bins, alpha=0.6, color=COLORS["nontarget"],
            label="non-target", density=True)
    ax.hist(cal[y_all==1], bins=bins, alpha=0.75, color=COLORS["target"],
            label="target", density=True)
    ax.axvline(0,       color=COLORS["gray"],  ls=":",  lw=2.5, label="thresh=0 ✓")
    ax.axvline(thr_cal, color=COLORS["green"], ls="--", lw=1.5, label=f"DCF thresh={thr_cal:.3f}")
    ax.set_title(f"{name} — calibrated\nEER={eer_cal*100:.1f}%", color=color)
    ax.set_xlabel("Calibrated log-odds")
    ax.legend(fontsize=7)

    # Reliability diagram (calibration curve)
    ax = axes[row_idx, 2]
    # Bin scores into deciles and plot empirical P(target) vs predicted P(target)
    from sklearn.calibration import calibration_curve
    prob_raw = 1 / (1 + np.exp(-raw))  # sigmoid of raw scores
    prob_cal = 1 / (1 + np.exp(-cal))  # sigmoid of calibrated scores
    n_bins_rel = min(8, int(y_all.sum() // 2))  # avoid too many bins with few positives
    try:
        frac_pos_raw, mean_pred_raw = calibration_curve(y_all, prob_raw, n_bins=n_bins_rel)
        frac_pos_cal, mean_pred_cal = calibration_curve(y_all, prob_cal, n_bins=n_bins_rel)
        ax.plot(mean_pred_raw, frac_pos_raw, "o--", color=COLORS["gray"],  lw=2, ms=6, label="raw")
        ax.plot(mean_pred_cal, frac_pos_cal, "o-",  color=color,            lw=2, ms=6, label="calibrated")
    except Exception:
        ax.text(0.5, 0.5, "too few positives\nfor reliability diagram",
                ha="center", va="center", transform=ax.transAxes, fontsize=9)
    ax.plot([0,1],[0,1],"k--",lw=1,label="perfect")
    ax.set_xlabel("Mean predicted P(target)")
    ax.set_ylabel("Empirical P(target)")
    ax.set_title(f"{name} — reliability diagram")
    ax.legend(fontsize=7)

plt.suptitle("Before vs after Platt calibration", fontsize=13)
plt.tight_layout()
plt.show()

## 5. After calibration — both systems on the same scale

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Overlay both calibrated distributions
ax = axes[0]
all_cal = np.concatenate([oof_audio_cal, oof_image_cal])
bins = np.linspace(all_cal.min(), all_cal.max(), 40)

for scores, name, color in [
    (oof_audio_cal, "Audio E008", COLORS["audio"]),
    (oof_image_cal, "Image E007", COLORS["image"]),
]:
    eer_c, _ = compute_eer(scores[y_all==1], scores[y_all==0])
    ax.hist(scores[y_all==0], bins=bins, alpha=0.35, color=color, density=True)
    ax.hist(scores[y_all==1], bins=bins, alpha=0.6,  color=color, density=True,
            label=f"{name} target (EER={eer_c*100:.1f}%)", histtype="step", lw=2)

ax.axvline(0, color="black", ls="--", lw=2, label="threshold = 0")
ax.set_xlabel("Calibrated log-odds score")
ax.set_title("Both systems — calibrated scores\n(now on the same scale)")
ax.legend(fontsize=8)

# Average of calibrated scores = simple fusion
ax = axes[1]
fused_cal = (oof_audio_cal + oof_image_cal) / 2
eer_fused, thr_fused = compute_eer(fused_cal[y_all==1], fused_cal[y_all==0])
dcf_fused, _         = compute_min_dcf(fused_cal[y_all==1], fused_cal[y_all==0])

bins = np.linspace(fused_cal.min(), fused_cal.max(), 40)
ax.hist(fused_cal[y_all==0], bins=bins, alpha=0.6, color=COLORS["nontarget"],
        label="non-target", density=True)
ax.hist(fused_cal[y_all==1], bins=bins, alpha=0.75, color=COLORS["target"],
        label="target", density=True)
ax.axvline(0,          color="black",         ls="--", lw=2, label="threshold = 0")
ax.axvline(thr_fused,  color=COLORS["green"], ls=":",  lw=2, label=f"DCF thresh={thr_fused:.3f}")
ax.set_xlabel("(audio_cal + image_cal) / 2")
ax.set_title(f"Simple average fusion (calibrated)\nEER={eer_fused*100:.2f}%, min-DCF={dcf_fused:.4f}")
ax.legend(fontsize=8)

plt.suptitle("Calibrated scores — ready for fusion", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Simple average fusion EER: {eer_fused*100:.2f}%  min-DCF: {dcf_fused:.4f}")
print(f"Threshold: {thr_fused:.4f}  (should be near 0)")
print(f"\nFor reference:")
eer_a, _ = compute_eer(oof_audio_cal[y_all==1], oof_audio_cal[y_all==0])
eer_i, _ = compute_eer(oof_image_cal[y_all==1], oof_image_cal[y_all==0])
print(f"  Audio alone (cal):  EER={eer_a*100:.2f}%")
print(f"  Image alone (cal):  EER={eer_i*100:.2f}%")
print(f"  Fusion (avg):       EER={eer_fused*100:.2f}%")

## 6. Save calibrators and OOF scores for fusion notebook

The fusion notebook (E009) will load these directly rather than re-running
the full CV loops.

In [ ]:
import pickle

cache_dir = Path("../cache")
cache_dir.mkdir(exist_ok=True)

calib_data = {
    "oof_audio_raw":  oof_audio,
    "oof_image_raw":  oof_image,
    "oof_audio_cal":  oof_audio_cal,
    "oof_image_cal":  oof_image_cal,
    "cal_audio":      cal_audio,
    "cal_image":      cal_image,
    "y_all":          y_all,
}

with open(cache_dir / "calibration.pkl", "wb") as f:
    pickle.dump(calib_data, f)

print("Saved to cache/calibration.pkl")
print("Contents:", list(calib_data.keys()))

## Summary

| System | Raw threshold | Calibrated threshold | EER (cal) |
|--------|--------------|---------------------|----------|
| Audio E008 | ≈ −0.078 | ≈ 0 | same |
| Image E007 | ≈ −5.028 | ≈ 0 | same |
| Fusion (avg) | — | ≈ 0 | see above |

EER doesn't change after calibration (it's threshold-free) — only the
threshold and scale change. The fusion notebook (E009) will use the
calibrated OOF scores to learn fusion weights.